# 08 · Cuándo una señal habría merecido ejecutarse

Construimos etiquetas de **horizonte fijo de un mes** para momentum y reversión. La pregunta es si una operación aislada habría obtenido más que mantener el capital en efectivo durante el mismo mes. Se aplican los costes y dividendos ya documentados.

La etiqueta central incluye negociación. Una única sensibilidad predefinida elimina la negociación **solo del etiquetado**, conservando dividendos, préstamo y efectivo. No escogemos entre ambas por sus resultados ni entrenamos modelos o contrastamos características en este cuaderno.

Antes de empezar, ejecuta los notebooks **01 → 02 → 07**, en ese orden. Este cuaderno utiliza sus tablas y no descarga ni prepara las fuentes.


## 1. Entradas y supuestos

La característica describe información anterior al objetivo, mientras que la etiqueta utiliza el resultado del mes objetivo y solo puede conocerse después. Los valores posteriores de dividendos y RF intervienen en el resultado contable, no en los predictores.

Las etiquetas son retrospectivas, es decir, los promedios de Shiller no son precios de ejecución y D/12 no es un pago mensual observado. El final del mes es el límite mínimo temporal del resultado, no una certificación de cuándo se publicaron sus datos.


In [ ]:
import pandas as pd
from IPython.display import display

from tfm.data import project_root, read_result, save_result
from tfm.features import FEATURE_COLUMNS
from tfm.labels import build_monthly_labels, summarize_labels, LABEL_TOLERANCE, SENSITIVITY_NAME

ROOT = project_root()
caracteristicas, _ = read_result(ROOT, "monthly_features")
componentes, _ = read_result(ROOT, "monthly_return_components")
senales, _ = read_result(ROOT, "shiller_signals")
parametros = {"initial_capital": 100.0, "transaction_cost_rate": 0.001,
              "annual_borrow_rate": 0.01, "include_dividends": True}
display(pd.DataFrame([parametros]).T.rename(columns={0: "Supuesto central"}))
print(f"Objetivos disponibles: {componentes.period.iloc[0]} → {componentes.period.iloc[-1]}")
print("Solo desarrollo 1927–1989. Cada operación aislada parte de 100 y termina liquidada.")


## 2. Las dos definiciones quedan fijadas antes del cálculo

**Central:** una compra recibe el dividendo aproximado, una venta corta paga ese dividendo y el préstamo. Se cobra negociación al abrir y cerrar. El efectivo obtiene RF. El capital del corto recibe RF según la misma convención del simulador, sin remunerar dos veces los ingresos de la venta corta.

**Sensibilidad:** cambia únicamente la tarifa de negociación a cero para construir la etiqueta. No eliminamos préstamo ni dividendos. Cuando se evalúe económicamente un filtro se cobrarán los costes del simulador, también para un modelo que use esta etiqueta alternativa.

La etiqueta es 1 si `retorno neto aislado − RF > 0` y 0 en caso contrario. En código utilizamos una tolerancia fija de `1e-12` en retorno decimal para tratar como empate el ruido numérico. No es un umbral económico elegido por rentabilidad. Los meses sin señal de operación conservan etiqueta ausente y no son negativos del clasificador.

Referencia de diseño: [Jansen, capítulo 7, ejemplo de etiquetas](https://github.com/stefan-jansen/machine-learning-for-trading/blob/91c903e71c1d0f1b9015f18551fadbb60bcad132/07_defining_the_learning_task/03_label_methods.py). Se adapta la estructura de señal más resultado futuro a nuestro motor contable y a la comparación con RF, no se copia su señal de veinte días ni sus datos de SPY. La procedencia y referencia bibliográfica preparada están en `docs/referencias_jansen.md`.


In [ ]:
etiquetas = build_monthly_labels(componentes, senales, parametros)
# Conservar todos los meses: una fila por regla y objetivo, con la misma información previa.
dataset = etiquetas.merge(
    caracteristicas, on=["asset", "target_period", "decision_period"],
    how="left", validate="many_to_one", indicator=True,
)
if not dataset["_merge"].eq("both").all():
    raise ValueError("Faltan características para algún mes; vuelve a ejecutar el notebook 07.")
dataset = dataset.drop(columns="_merge")
dataset["training_ready"] = dataset.features_ready & dataset.has_signal
resumen_reglas = summarize_labels(dataset)
display(pd.DataFrame(resumen_reglas).set_index("strategy"))
print("Los ejemplos de entrenamiento requieren características completas y una señal distinta de efectivo.")
print("Tolerancia numérica en retorno decimal:", LABEL_TOLERANCE)


## 3. Una operación de cada regla explicada paso a paso

Puedes cambiar `mes_objetivo` para inspeccionar otra decisión. El balance muestra qué se gana o paga y compara el resultado con dejar las mismas 100 unidades en efectivo. La etiqueta alternativa excluye negociación, pero no modifica los demás supuestos. Una ganancia absoluta puede recibir etiqueta 0 si no supera a RF.


In [ ]:
mes_objetivo = "1975-05"
if mes_objetivo not in set(dataset.target_period):
    raise ValueError("Escoge un objetivo entre 1927-01 y 1989-12")
columnas_cuenta = [
    "strategy", "decision_period", "target_period", "position", "capital_start",
    "transaction_cost", "price_pnl", "dividend_pnl", "cash_interest", "borrow_cost",
    "liquidation_cost", "capital_end", "cash_capital_end", "excess_return", "label",
    "capital_end_no_transaction_cost", "label_no_transaction_cost", "label_status", "training_ready",
]
display(dataset.loc[dataset.target_period.eq(mes_objetivo), columnas_cuenta].T)
print("Etiqueta 1: supera al efectivo. Etiqueta 0: no lo supera. Ausente: la regla no propone operar.")


## 4. Cuánto cambia el etiquetado por la negociación

La tabla separa el calendario completo, las operaciones propuestas y los ejemplos con características completas. El denominador de la fracción de cambios es el número de ejemplos de esa regla que cumplen ambos requisitos, no los 756 meses.

Esta comparación describe sensibilidad de las etiquetas. No prueba capacidad predictiva ni demuestra que una variante produzca una estrategia mejor. Las dos definiciones se registran como alternativas consideradas y cualquier entrenamiento posterior con ellas contará en el registro científico de variantes.


In [ ]:
comparacion = pd.DataFrame(resumen_reglas)[[
    "strategy", "training_examples", "training_positive", "training_negative",
    "changed_labels_training", "changed_fraction_training",
]].set_index("strategy")
display(comparacion)
cambios = dataset.loc[dataset.training_ready & dataset.label_changed.fillna(False)]
display(cambios[["strategy", "target_period", "position", "excess_return",
                 "excess_return_no_transaction_cost", "label", "label_no_transaction_cost"]].head(10))
print(f"Ejemplos de cambios mostrados: {min(10, len(cambios))}; cambios totales en la muestra utilizable: {len(cambios)}")


## 5. Qué columnas podrá utilizar el modelo

**Solo `FEATURE_COLUMNS` contiene predictores.** La señal original conserva su función en cada regla. Las etiquetas, los retornos del mes objetivo, los costes realizados y los capitales finales son resultados: nunca se seleccionarán automáticamente todas las columnas numéricas como entrada.

`features_ready` marca cobertura de las diez características. `training_ready` añade que hay una operación propuesta. Ambas son marcas de control. Las 749 filas mensuales completas pueden dar un número diferente de ejemplos para cada regla. El calentamiento de las futuras transformaciones puede reducirlo más.


In [ ]:
print("Predictores:", FEATURE_COLUMNS)
print("Objetivo central: label | Sensibilidad:", SENSITIVITY_NAME, "→ label_no_transaction_cost")
display(dataset[["strategy", "decision_period", "target_period", "label_end_period",
                 "has_signal", "features_ready", "training_ready", "label", "label_no_transaction_cost"]].head(12))


## 6. Guardar el conjunto etiquetado

**Esta celda guarda o sustituye** `data/processed/labelled_signals.parquet` y su JSON. Conserva los 756 meses por cada una de las dos reglas, con etiquetas ausentes cuando no hay operación. No guarda dos conjuntos con filas diferentes para las dos convenciones.

La estrategia final mantendrá posiciones entre meses cuando corresponda. No se deben capitalizar los retornos aislados para presentar una curva de esa estrategia. El beneficio de este objetivo sencillo de aprendizaje se comprobará después con el simulador completo.

Regenerar precios, señales, componentes o características retira este conjunto. Las hipótesis detalladas y el protocolo de cribado se cerrarán antes de calcular relaciones predictivas. Este cuaderno no selecciona características, umbrales o modelos.


In [ ]:
informe = {
    "first_target": dataset.target_period.min(), "last_target": dataset.target_period.max(),
    "parameters": parametros, "feature_columns": FEATURE_COLUMNS,
    "central_target": "label", "alternative_target": "label_no_transaction_cost",
    "label_tolerance": LABEL_TOLERANCE,
    "definitions": [
        {"name": "central", "transaction_cost_rate": parametros["transaction_cost_rate"],
         "criterion": "isolated_net_return - rf_return > numeric_tolerance"},
        {"name": SENSITIVITY_NAME, "transaction_cost_rate": 0.0,
         "criterion": "isolated_net_return_without_trading_cost - rf_return > numeric_tolerance"},
    ],
    "design_reference": "Jansen, chapter 7, 03_label_methods.py, revision 91c903e71c1d0f1b9015f18551fadbb60bcad132; own accounting adaptation",
    "strategies": resumen_reglas,
    "warnings": [
        "Monthly means and reconstructed dividends: retrospective approximation, not executable trades.",
        "Each label starts in cash and liquidates; portfolio evaluation can carry positions between months.",
        "No-trade signals are not negative labels. Features alone are model inputs.",
        "Label end period is a temporal lower bound, not proof of historical publication availability.",
        "Only trading costs are removed in the sensitivity label; evaluation retains trading costs.",
        "No predictive screening, training, model selection or post-1989 evaluation has been performed.",
    ],
}
resumen = save_result(ROOT, "labelled_signals", dataset, informe)
print("Guardado:", resumen["path"])
print(f"Filas de calendario: {resumen['rows']}; dos reglas con etiquetas y sensibilidad sobre los mismos meses.")
